In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, precision_recall_curve, auc, confusion_matrix

# Load preprocessed data
df = pd.read_csv('../data/processed/fraud_data_engineered.csv')

# Drop non-predictive high-cardinality IDs and raw timestamps
X = df.drop(columns=['class', 'user_id', 'device_id', 'signup_time', 'purchase_time', 'ip_address', 'ip_int'])
y = df['class']

# Define feature subgroups
num_features = ['purchase_value', 'age', 'time_since_signup', 'hour_of_day', 'day_of_week']
cat_features = ['source', 'browser', 'sex', 'country']

# Stratified Split to preserve real world imbalance in test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)
print(df.columns.tolist())


['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class', 'ip_int', 'country', 'time_since_signup', 'hour_of_day', 'day_of_week']


In [2]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)



In [4]:
# Resampling Strategy: Apply SMOTE only to the training set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)

# --- 1. Baseline Model (Logistic Regression) ---
baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train_res, y_train_res)
y_pred_base = baseline.predict(X_test_proc)
y_probs_base = baseline.predict_proba(X_test_proc)[:, 1]

# --- 2. Advanced Ensemble Model (XGBoost) ---
xgb_model = XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=1, random_state=42)
xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test_proc)
y_probs_xgb = xgb_model.predict_proba(X_test_proc)[:, 1]

# --- Evaluation Function for PR Curves ---
def evaluate_model_performance(y_true, y_probs, y_pred, name):
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    auc_pr = auc(recall, precision)
    print(f"=== {name} Performance ===")
    print(f"AUC-PR Score: {auc_pr:.4f}")
    print(classification_report(y_true, y_pred))
    return precision, recall, auc_pr

p_b, r_b, auc_b = evaluate_model_performance(y_test, y_probs_base, y_pred_base, "Logistic Regression Baseline")
p_x, r_x, auc_x = evaluate_model_performance(y_test, y_probs_xgb, y_pred_xgb, "XGBoost Classifier")

# Save the best model and transformation configuration
joblib.dump(xgb_model, '../models/best_xgb_model.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')

=== Logistic Regression Baseline Performance ===
AUC-PR Score: 0.4058
              precision    recall  f1-score   support

           0       0.95      0.66      0.78     27393
           1       0.17      0.69      0.27      2830

    accuracy                           0.66     30223
   macro avg       0.56      0.67      0.53     30223
weighted avg       0.88      0.66      0.73     30223

=== XGBoost Classifier Performance ===
AUC-PR Score: 0.6074
              precision    recall  f1-score   support

           0       0.95      1.00      0.98     27393
           1       0.97      0.52      0.68      2830

    accuracy                           0.95     30223
   macro avg       0.96      0.76      0.83     30223
weighted avg       0.95      0.95      0.95     30223



['../models/preprocessor.pkl']